In [0]:
#!pip install langgraph
#!pip install optuna
#!pip install xgboost

In [0]:
import mlflow
import mlflow.xgboost
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
import optuna
import xgboost as xgb  # <--- IMPORTAÇÃO ADICIONADA AQUI

from langgraph.graph import StateGraph, END
from typing import TypedDict, Any

# -------------------------------------------------------------------------
# 1. DEFINIÇÃO DAS FERRAMENTAS (TOOLS)
# -------------------------------------------------------------------------

def tool_feature_engineering_and_split(catalog_schema: str = "emissor_a.gold.behavior_score_clientes"):
    """Gera dados sintéticos com forte correlação de risco de crédito (padrões reais)."""
    print(f"[Ferramenta] Gerando dados sintéticos com correlação de risco para: {catalog_schema}")
    
    np.random.seed(42)
    n = 2500  # Aumentando um pouco a massa de dados para o treino
    
    # 1. Gerando variáveis preditivas com distribuições realistas de crédito
    max_dias_atraso = np.random.exponential(scale=4, size=n).astype(int)
    qtd_faturas_atrasadas = np.random.poisson(lam=0.7, size=n)
    media_utilizacao = np.random.beta(a=2, b=4, size=n)  # Concentração típica de uso de limite
    qtd_vezes_parcial = np.random.poisson(lam=0.4, size=n)
    total_transacoes = np.random.randint(5, 50, size=n)
    
    # 2. Criando a lógica de negócio (Score de Risco Latente)
    # Quanto maiores os atrasos, utilização e pagamentos parciais, maior o risco
    latent_score = (
        0.15 * max_dias_atraso + 
        0.60 * qtd_faturas_atrasadas + 
        2.50 * media_utilizacao + 
        0.90 * qtd_vezes_parcial + 
        np.random.normal(0, 0.4, size=n)  # Ruído gaussiano leve
    )
    
    # 3. Transformando em probabilidade via função Sigmoid (ajustando taxa de default para ~15%)
    probabilities = 1 / (1 + np.exp(-latent_score + 3.0))
    
    # 4. Definindo a flag de default de forma correlacionada
    flag_default = (np.random.uniform(0, 1, size=n) < probabilities).astype(int)
    
    df = pd.DataFrame({
        "cliente_id": [f"C{i:04d}" for i in range(n)],
        "max_dias_atraso_historico": max_dias_atraso,
        "qtd_faturas_atrasadas": qtd_faturas_atrasadas,
        "media_utilizacao_limite": media_utilizacao,
        "qtd_vezes_pagamento_parcial": qtd_vezes_parcial,
        "total_transacoes_periodo": total_transacoes,
        "flag_default_90d": flag_default
    })
    
    print(f"[Ferramenta] Taxa real de inadimplência gerada na base: {df['flag_default_90d'].mean():.2%}")
    
    X = df.drop(columns=["cliente_id", "flag_default_90d"])
    y = df["flag_default_90d"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print("[Ferramenta] Feature Engineering e Split concluídos com sucesso.")
    return {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}

def tool_hyperparameter_tuning_and_train(data_dict, n_trials=3):
    """Executa a otimização de hiperparâmetros com Optuna e loga no MLflow."""
    X_train, X_test, y_train, y_test = data_dict["X_train"], data_dict["X_test"], data_dict["y_train"], data_dict["y_test"]
    
    mlflow.set_experiment("/Shared/emissor_a_behavior_score")
    
    def objective(trial):
        params = {
            "max_depth": trial.suggest_int("max_depth", 3, 6),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 50, 150),
            "random_state": 42
        }
        
        with mlflow.start_run(nested=True):
            model = xgb.XGBClassifier(**params)
            model.fit(X_train, y_train)
            preds = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, preds)
            mlflow.log_params(params)
            mlflow.log_metric("val_auc", auc)
            
        return auc

    print("[Ferramenta] Iniciando tuning de hiperparâmetros com Optuna...")
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    
    best_params = study.best_params
    print(f"[Ferramenta] Melhores parâmetros encontrados: {best_params}")
    
    best_model = xgb.XGBClassifier(**best_params, random_state=42)
    best_model.fit(X_train, y_train)
    
    return {"model": best_model, "best_params": best_params, "best_auc": study.best_value}

def tool_rigorous_validation(model_results, data_dict, min_auc_threshold=0.60):
    """Aplica Quality Gates baseados em métricas (AUC e KS)."""
    model = model_results["model"]
    X_test, y_test = data_dict["X_test"], data_dict["y_test"]
    
    preds_proba = model.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test, preds_proba)
    
    df_eval = pd.DataFrame({"prob": preds_proba, "target": y_test})
    class_0 = df_eval[df_eval["target"] == 0]["prob"]
    class_1 = df_eval[df_eval["target"] == 1]["prob"]
    ks_stat, _ = ks_2samp(class_0, class_1)
    
    print(f"[Ferramenta] Validação - AUC: {auc_score:.4f} | KS: {ks_stat:.4f}")
    
    approved = bool(auc_score >= min_auc_threshold)
    return {"approved": approved, "auc": auc_score, "ks": ks_stat}

from mlflow.models.signature import infer_signature

def tool_register_and_deploy_model(model_results, validation_results, data_dict, model_name="emissor_a_behavior_score"):
    """Registra o modelo no Unity Catalog / MLflow com a assinatura obrigatória."""
    if not validation_results["approved"]:
        raise ValueError("Tentativa de deploy de um modelo reprovado.")
        
    model = model_results["model"]
    X_train = data_dict["X_train"]
    
    # 1. Inferir a assinatura usando os dados de treino (entrada e saída prevista)
    predictions = model.predict(X_train)
    signature = infer_signature(X_train, predictions)
    
    with mlflow.start_run() as run:
        mlflow.log_metrics({"final_auc": validation_results["auc"], "final_ks": validation_results["ks"]})
        
        # 2. Passar a assinatura no log do modelo
        mlflow.xgboost.log_model(
            model, 
            artifact_path="model",
            signature=signature,
            registered_model_name=model_name
        )
        
        model_uri = f"runs:/{run.info.run_id}/model"
        print(f"[Ferramenta] Modelo com assinatura registrada com sucesso no Unity Catalog! URI: {model_uri}")
        
    return {"status": "deployed", "model_uri": model_uri}


# -------------------------------------------------------------------------
# 2. CONSTRUÇÃO DO GRAFO (LANGGRAPH)
# -------------------------------------------------------------------------

class AgentState(TypedDict):
    data: Any
    model_results: Any
    validation: Any
    status: str

workflow = StateGraph(AgentState)

workflow.add_node("feature_engineering", lambda state: {"data": tool_feature_engineering_and_split(), "status": "engineered"})
workflow.add_node("training", lambda state: {"model_results": tool_hyperparameter_tuning_and_train(state["data"]), "status": "trained"})
workflow.add_node("validation", lambda state: {"validation": tool_rigorous_validation(state["model_results"], state["data"]), "status": "validated"})
workflow.add_node("deployment", lambda state: {"deploy": tool_register_and_deploy_model(state["model_results"], state["validation"], state["data"]), "status": "deployed"})

workflow.set_entry_point("feature_engineering")
workflow.add_edge("feature_engineering", "training")
workflow.add_edge("training", "validation")

def should_deploy(state):
    return "deploy" if state["validation"]["approved"] else "stop"

workflow.add_conditional_edges("validation", should_deploy, {"deploy": "deployment", "stop": END})
workflow.add_edge("deployment", END)

app = workflow.compile()
print("[Agente] LangGraph compilado com sucesso e ferramentas carregadas!")

# -------------------------------------------------------------------------
# 3. EXECUÇÃO DO AGENTE
# -------------------------------------------------------------------------
estado_inicial = {"data": None, "model_results": None, "validation": None, "status": "iniciando"}

print("\n=== Disparando o Agente Autônomo de Data Science ===")
resposta_final = app.invoke(estado_inicial)

print("\n=== Execução Finalizada! ===")
print(f"Status final do agente: {resposta_final.get('status')}")
if resposta_final.get("validation"):
    print(f"Métricas do modelo: {resposta_final['validation']}")

In [0]:
def tool_monitor_data_drift(data_dict, threshold=0.25):
    """Calcula o PSI (Population Stability Index) entre a base de treino e uma base de produção simulada."""
    print("[Ferramenta] Monitorando Data Drift (PSI)...")
    X_train = data_dict["X_train"]
    
    # Simulando uma base de dados que chega em produção (com leve desvio na utilização de limite)
    np.random.seed(99)
    production_sample = X_train.copy()
    production_sample["media_utilizacao_limite"] = production_sample["media_utilizacao_limite"] * np.random.uniform(0.9, 1.2, size=len(production_sample))
    
    # Calculando o PSI para a principal feature ("media_utilizacao_limite")
    expected = X_train["media_utilizacao_limite"].values
    observed = production_sample["media_utilizacao_limite"].values
    
    def calculate_psi(exp, obs, num_bins=10):
        bins = np.percentile(exp, np.linspace(0, 100, num_bins + 1))
        bins[0] = -np.inf
        bins[-1] = np.inf
        freq_esp, _ = np.histogram(exp, bins=bins)
        freq_obs, _ = np.histogram(obs, bins=bins)
        pct_esp = np.maximum(freq_esp / len(exp), 1e-4)
        pct_obs = np.maximum(freq_obs / len(obs), 1e-4)
        return float(np.sum((pct_obs - pct_esp) * np.log(pct_obs / pct_esp)))
    
    psi_value = calculate_psi(expected, observed)
    print(f"[Ferramenta] PSI calculado: {psi_value:.4f}")
    
    drift_detected = bool(psi_value > threshold)
    return {"drift_detected": drift_detected, "psi": psi_value}

def tool_configure_ab_test(validation_results, drift_results, model_name="emissor_a_behavior_score"):
    """Configura o roteamento de tráfego A/B (Champion/Challenger) se o modelo estiver apto."""
    if not validation_results["approved"] or drift_results["drift_detected"]:
        print("[Ferramenta] Teste A/B cancelado: Modelo reprovado ou Data Drift detectado.")
        return {"ab_status": "aborted"}
        
    print(f"[Ferramenta] Configurando Teste A/B para o modelo '{model_name}':")
    print(" - Modelo Atual (Champion): 90% do tráfego")
    print(" - Novo Modelo (Challenger): 10% do tráfego")
    
    # Simulação da chamada de atualização do endpoint de Serving no Databricks:
    # w.serving_endpoints.update(...)
    
    return {"ab_status": "configured", "champion_traffic": 90, "challenger_traffic": 10}

In [0]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Any

class AgentState(TypedDict):
    data: Any
    model_results: Any
    validation: Any
    drift_results: Any
    deploy: Any
    ab_test: Any
    status: str

workflow = StateGraph(AgentState)

# Adicionando todos os nós do ciclo de vida
workflow.add_node("feature_engineering", lambda state: {"data": tool_feature_engineering_and_split(), "status": "engineered"})
workflow.add_node("training", lambda state: {"model_results": tool_hyperparameter_tuning_and_train(state["data"]), "status": "trained"})
workflow.add_node("validation", lambda state: {"validation": tool_rigorous_validation(state["model_results"], state["data"]), "status": "validated"})
workflow.add_node("drift_monitoring", lambda state: {"drift_results": tool_monitor_data_drift(state["data"]), "status": "drift_checked"})
workflow.add_node("deployment", lambda state: {"deploy": tool_register_and_deploy_model(state["model_results"], state["validation"], state["data"]), "status": "deployed"})
workflow.add_node("ab_testing", lambda state: {"ab_test": tool_configure_ab_test(state["validation"], state["drift_results"]), "status": "completed"})

# Definindo o fluxo linear e condicional
workflow.set_entry_point("feature_engineering")
workflow.add_edge("feature_engineering", "training")
workflow.add_edge("training", "validation")

# Condicional 1: Só faz drift e deploy se passar na validação
def check_validation(state):
    return "pass" if state["validation"]["approved"] else "stop"

workflow.add_conditional_edges("validation", check_validation, {"pass": "drift_monitoring", "stop": END})

workflow.add_edge("drift_monitoring", "deployment")
workflow.add_edge("deployment", "ab_testing")
workflow.add_edge("ab_testing", END)

app = workflow.compile()
print("[Agente MLOps] Pipeline completo de ponta a ponta compilado com sucesso!")

In [0]:
estado_inicial = {
    "data": None, "model_results": None, "validation": None, 
    "drift_results": None, "deploy": None, "ab_test": None, "status": "iniciando"
}

print("=== Disparando o Agente Autônomo de MLOps (Ponta a Ponta) ===")
resposta_final = app.invoke(estado_inicial)

print("\n=== Ciclo Concluído com Sucesso! ===")
print(f"Status Final: {resposta_final.get('status')}")
print(f"Validação (AUC/KS): {resposta_final.get('validation')}")
print(f"Monitoramento de Drift (PSI): {resposta_final.get('drift_results')}")
print(f"Status do Teste A/B: {resposta_final.get('ab_test')}")